# AraseのDSI座標とFAC座標系の関係を理解

# FAC座標系の定義
- z軸は、背景磁場$B_{0}$の単位ベクトルで与える。
- x軸は、反地球方向かつz軸と垂直な単位ベクトルで与える。
- y軸は、z軸とx軸の外積で与える。

# データ保存先

In [ ]:
import os
os.environ["SPEDAS_DATA_DIR"] = "/mnt/j/observation_data/"

# 反地球方向単位ベクトル (GSM座標系, DSI座標系) $\hat{\mathbf{r}}_{\mathrm{GSM/DSI}}$の決定

In [ ]:
import pyspedas as psp
import pytplot as pt
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import xarray as xr

pt.del_data('*')

trange = ['2022-09-01/21:00', '2022-09-01/23:59']

psp.erg.orb(trange=trange, level='l2', datatype='def', no_update=True)
da_Arase_pos_gsm    = pt.data_quants['erg_orb_l2_pos_gsm']
da_Arase_pos_gsm    = da_Arase_pos_gsm.sortby('time').sel(time=slice(trange[0], trange[1]))

print(da_Arase_pos_gsm)

In [ ]:
da_Arase_pos_unit_gsm   = da_Arase_pos_gsm / np.sqrt((da_Arase_pos_gsm * da_Arase_pos_gsm).sum(dim='v_dim'))
print(da_Arase_pos_unit_gsm)

In [ ]:
pt.store_data('Arase_pos_unit_gsm', data={'x': da_Arase_pos_unit_gsm.time, 'y': da_Arase_pos_unit_gsm.data})
psp.cotrans(name_in='Arase_pos_unit_gsm', name_out='Arase_pos_unit_j2000', coord_in='gsm', coord_out='j2000')
psp.erg.erg_cotrans(in_name='Arase_pos_unit_j2000', out_name='Arase_pos_unit_dsi', in_coord='j2000', out_coord='dsi')

da_Arase_pos_unit_dsi   = pt.data_quants['Arase_pos_unit_dsi']

print(da_Arase_pos_unit_dsi)

# 背景磁場単位ベクトル (DSI座標系) $\hat{\mathbf{b}}_{0, \mathrm{DSI}}$ の決定

In [ ]:
import pyspedas as psp
import pytplot as pt
import xarray as xr
import numpy as np

psp.erg.mgf(trange=trange, level='l2', datatype='64hz', coord='dsi', no_update=True)

da_B_64Hz   = pt.data_quants['erg_mgf_l2_mag_64hz_dsi'].sortby('time').sel(time=slice(trange[0], trange[1]))

In [ ]:
background_time_sec = 100 #[sec]

In [ ]:
time_width_B_64Hz       = (da_B_64Hz.time[10] - da_B_64Hz.time[9]) / np.timedelta64(1, 's')
da_B_background         = da_B_64Hz.rolling(time=int(background_time_sec / time_width_B_64Hz), center=True).mean('time')
da_B_background_unit    = da_B_background / np.sqrt((da_B_background * da_B_background).sum(dim='v_dim'))
da_B_background_unit    = da_B_background_unit.dropna(how='all', dim='time')

print(da_B_background_unit)

# DSI座標系で、FACの単位ベクトルを定義
```math
\mathbf{u} = \hat{\mathbf{r}}_{\mathrm{DSI}} - (\hat{\mathbf{r}}_{\mathrm{DSI}} \cdot \hat{\mathbf{b}}_{0, \mathrm{DSI}}) \hat{\mathbf{b}}_{0, \mathrm{DSI}}    \\
\mathbf{e}_{z, \mathrm{FAC}}    := \hat{\mathbf{b}}_{0, \mathrm{DSI}}   \\
\mathbf{e}_{x, \mathrm{FAC}}    := \frac{\mathbf{u}}{|\mathbf{u}|}      \\
\mathbf{e}_{y, \mathrm{FAC}}    := \mathbf{e}_{z, \mathrm{FAC}} \times \mathbf{e}_{x, \mathrm{FAC}}
```

In [ ]:
time_array  = da_B_background_unit.time

da_Arase_pos_unit_dsi_interp    = da_Arase_pos_unit_dsi.interp(time=time_array)

da_u_   = da_Arase_pos_unit_dsi_interp - (da_Arase_pos_unit_dsi_interp * da_B_background_unit).sum(dim='v_dim') * da_B_background_unit

da_e_z_FAC_inDSI    = da_B_background_unit.drop_attrs()
da_e_x_FAC_inDSI    = (da_u_ / np.sqrt((da_u_ * da_u_).sum(dim='v_dim'))).drop_attrs()
da_e_y_FAC_inDSI    = (xr.apply_ufunc(np.cross, da_e_z_FAC_inDSI, da_e_x_FAC_inDSI, input_core_dims=[['v_dim'], ['v_dim']], output_core_dims=[['v_dim']], vectorize=True)).drop_attrs()

print(da_e_x_FAC_inDSI)
print('')
print(da_e_y_FAC_inDSI)
print('')
print(da_e_z_FAC_inDSI)

# FAC座標系にDSI単位ベクトルを変換

In [ ]:
import xarray as xr
import numpy as np

# FAC の各軸を DSI で表したものを 3×3 行列にまとめる
A_fac_in_dsi = xr.concat(
    [da_e_x_FAC_inDSI, da_e_y_FAC_inDSI, da_e_z_FAC_inDSI],
    dim='axis'
)
A_fac_in_dsi = A_fac_in_dsi.assign_coords(axis=['x_fac', 'y_fac', 'z_fac'])

print(A_fac_in_dsi)
# dims: (time, axis, v_dim)  = (time, FAC軸, DSI成分)

# DSI→FAC の回転行列（直交行列なので転置）
A_dsi_to_fac = A_fac_in_dsi.transpose('time', 'v_dim', 'axis')

print(A_dsi_to_fac)

In [ ]:
# DSI-x を FAC で表したもの（成分: FAC-x, FAC-y, FAC-z）
da_e_X_DSI_inFAC = A_dsi_to_fac.sel(v_dim=0)  # or v_dim='x'
da_e_Y_DSI_inFAC = A_dsi_to_fac.sel(v_dim=1)  # or v_dim='y'
da_e_Z_DSI_inFAC = A_dsi_to_fac.sel(v_dim=2)  # or v_dim='z'

In [ ]:
print(da_e_Z_DSI_inFAC.dims)

In [ ]:
cos_theta_z = da_e_Z_DSI_inFAC.isel(axis=2)
theta_z_deg = np.rad2deg(np.arccos(cos_theta_z))

import matplotlib.pyplot as plt
plt.figure(figsize=(10,10))
theta_z_deg.plot()
plt.minorticks_on()
plt.ylabel('angle between DSI-Z and FAC-Z [deg]')
plt.grid(True, which='both', linestyle=':')

In [ ]:
import matplotlib.pyplot as plt

it = 400000  # 好きな time index
ex = da_e_X_DSI_inFAC.isel(time=it)
ey = da_e_Y_DSI_inFAC.isel(time=it)
ez = da_e_Z_DSI_inFAC.isel(time=it)

plt.figure(figsize=(6, 5))
# FAC の基底
plt.quiver(0, 0, 1, 0, angles='xy', scale_units='xy', scale=1, color='k', label='FAC-x')
plt.quiver(0, 0, 0, 1, angles='xy', scale_units='xy', scale=1, color='gray', label='FAC-y')

# DSI の x,y 軸を FAC 平面に投影
plt.quiver(0, 0,
           ex.isel(axis=0), ex.isel(axis=1),
           angles='xy', scale_units='xy', scale=1, color='r', label='DSI-x')
plt.quiver(0, 0,
           ey.isel(axis=0), ey.isel(axis=1),
           angles='xy', scale_units='xy', scale=1, color='b', label='DSI-y')
plt.quiver(0, 0,
           ez.isel(axis=0), ez.isel(axis=1),
           angles='xy', scale_units='xy', scale=1, color='green', label='DSI-z')

plt.axhline(0, color='k', linewidth=0.5)
plt.axvline(0, color='k', linewidth=0.5)
plt.gca().set_aspect('equal', adjustable='box')
plt.xlim(-1, 1)
plt.ylim(-1, 1)
plt.legend()
plt.xlabel('FAC-x (Radial)')
plt.ylabel('FAC-y (Longitudinal)')
plt.minorticks_on()
plt.grid(True, which='both', linestyle=':')
plt.title(str(da_e_X_DSI_inFAC.time.values[it]))
plt.grid(True)


In [ ]:
import matplotlib.pyplot as plt

it = 400000
ex = da_e_X_DSI_inFAC.isel(time=it)  # (axis: x_fac,y_fac,z_fac)
ey = da_e_Y_DSI_inFAC.isel(time=it)
ez = da_e_Z_DSI_inFAC.isel(time=it)

# FAC 成分を取り出しやすくしておく
ex_x = ex.sel(axis='x_fac').item()
ex_y = ex.sel(axis='y_fac').item()
ex_z = ex.sel(axis='z_fac').item()

ey_x = ey.sel(axis='x_fac').item()
ey_y = ey.sel(axis='y_fac').item()
ey_z = ey.sel(axis='z_fac').item()

ez_x = ez.sel(axis='x_fac').item()
ez_y = ez.sel(axis='y_fac').item()
ez_z = ez.sel(axis='z_fac').item()

fig, axs = plt.subplots(1, 3, figsize=(15, 5))  # 6x5 を 3 枚分

def setup_ax(ax, xlabel, ylabel, title):
    ax.axhline(0, color='k', linewidth=0.5)
    ax.axvline(0, color='k', linewidth=0.5)
    ax.set_aspect('equal', adjustable='box')
    ax.set_xlim(-1, 1)
    ax.set_ylim(-1, 1)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.minorticks_on()
    ax.grid(True, which='both', linestyle=':')
    ax.set_title(title)

# ---------------- (x, y) plane ----------------
ax = axs[0]
# FAC 基底
ax.quiver(0, 0, 1, 0, angles='xy', scale_units='xy', scale=1, color='k',   label='FAC-x')
ax.quiver(0, 0, 0, 1, angles='xy', scale_units='xy', scale=1, color='gray', label='FAC-y')
ax.quiver(0, 0, 0, 0, angles='xy', scale_units='xy', scale=1, color='purple', label='FAC-z', linewidth=0)
# DSI 軸
ax.quiver(0, 0, ex_x, ex_y, angles='xy', scale_units='xy', scale=1, color='r', label='DSI-x')
ax.quiver(0, 0, ey_x, ey_y, angles='xy', scale_units='xy', scale=1, color='b', label='DSI-y')
ax.quiver(0, 0, ez_x, ez_y, angles='xy', scale_units='xy', scale=1, color='g', label='DSI-z')

setup_ax(ax, 'FAC-x (Radial)', 'FAC-y (Longitudinal)', '(x, y) plane')
ax.legend(loc='lower left')

# ---------------- (x, z) plane ----------------
ax = axs[1]
ax.quiver(0, 0, 1, 0, angles='xy', scale_units='xy', scale=1, color='k',   label='FAC-x')
ax.quiver(0, 0, 0, 1, angles='xy', scale_units='xy', scale=1, color='purple', label='FAC-z')

ax.quiver(0, 0, ex_x, ex_z, angles='xy', scale_units='xy', scale=1, color='r', label='DSI-x')
ax.quiver(0, 0, ey_x, ey_z, angles='xy', scale_units='xy', scale=1, color='b', label='DSI-y')
ax.quiver(0, 0, ez_x, ez_z, angles='xy', scale_units='xy', scale=1, color='g', label='DSI-z')

setup_ax(ax, 'FAC-x (Radial)', 'FAC-z (Parallel)', '(x, z) plane')

# ---------------- (y, z) plane ----------------
ax = axs[2]
ax.quiver(0, 0, 1, 0, angles='xy', scale_units='xy', scale=1, color='gray',   label='FAC-y')
ax.quiver(0, 0, 0, 1, angles='xy', scale_units='xy', scale=1, color='purple', label='FAC-z')

ax.quiver(0, 0, ex_y, ex_z, angles='xy', scale_units='xy', scale=1, color='r', label='DSI-x')
ax.quiver(0, 0, ey_y, ey_z, angles='xy', scale_units='xy', scale=1, color='b', label='DSI-y')
ax.quiver(0, 0, ez_y, ez_z, angles='xy', scale_units='xy', scale=1, color='g', label='DSI-z')

setup_ax(ax, 'FAC-y (Longitudinal)', 'FAC-z (Parallel)', '(y, z) plane')

fig.suptitle(str(da_e_X_DSI_inFAC.time.values[it]), fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
import os
import matplotlib.pyplot as plt

path_base_save_plot = (
    "/mnt/j/KAW_observation/E_B_ratio_Arase/2022-09-01/2230-2330/coordinate"
)
os.makedirs(path_base_save_plot, exist_ok=True)

def setup_ax(ax, xlabel, ylabel, title):
    ax.axhline(0, color='k', linewidth=0.5)
    ax.axvline(0, color='k', linewidth=0.5)
    ax.set_aspect('equal', adjustable='box')
    ax.set_xlim(-1, 1)
    ax.set_ylim(-1, 1)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.minorticks_on()
    ax.grid(True, which='both', linestyle=':')
    ax.set_title(title)

def plot_fac_dsi_frame(it, frame_idx, save_dir):
    ex = da_e_X_DSI_inFAC.isel(time=it)  # (axis: x_fac,y_fac,z_fac)
    ey = da_e_Y_DSI_inFAC.isel(time=it)
    ez = da_e_Z_DSI_inFAC.isel(time=it)

    # FAC 成分
    ex_x = ex.sel(axis='x_fac').item()
    ex_y = ex.sel(axis='y_fac').item()
    ex_z = ex.sel(axis='z_fac').item()

    ey_x = ey.sel(axis='x_fac').item()
    ey_y = ey.sel(axis='y_fac').item()
    ey_z = ey.sel(axis='z_fac').item()

    ez_x = ez.sel(axis='x_fac').item()
    ez_y = ez.sel(axis='y_fac').item()
    ez_z = ez.sel(axis='z_fac').item()

    fig, axs = plt.subplots(1, 3, figsize=(15, 5))

    # ------------- (x, y) plane -------------
    ax = axs[0]
    ax.quiver(0, 0, 1, 0, angles='xy', scale_units='xy', scale=1,
              color='k',   label='FAC-x')
    ax.quiver(0, 0, 0, 1, angles='xy', scale_units='xy', scale=1,
              color='gray', label='FAC-y')
    ax.quiver(0, 0, 0, 0, angles='xy', scale_units='xy', scale=1,
              color='purple', label='FAC-z', linewidth=0)

    ax.quiver(0, 0, ex_x, ex_y, angles='xy', scale_units='xy', scale=1,
              color='r', label='DSI-x')
    ax.quiver(0, 0, ey_x, ey_y, angles='xy', scale_units='xy', scale=1,
              color='b', label='DSI-y')
    ax.quiver(0, 0, ez_x, ez_y, angles='xy', scale_units='xy', scale=1,
              color='g', label='DSI-z')

    setup_ax(ax, 'FAC-x (Radial)', 'FAC-y (Longitudinal)', '(x, y) plane')
    ax.legend(loc='lower left')

    # ------------- (x, z) plane -------------
    ax = axs[1]
    ax.quiver(0, 0, 1, 0, angles='xy', scale_units='xy', scale=1,
              color='k',   label='FAC-x')
    ax.quiver(0, 0, 0, 1, angles='xy', scale_units='xy', scale=1,
              color='purple', label='FAC-z')

    ax.quiver(0, 0, ex_x, ex_z, angles='xy', scale_units='xy', scale=1,
              color='r', label='DSI-x')
    ax.quiver(0, 0, ey_x, ey_z, angles='xy', scale_units='xy', scale=1,
              color='b', label='DSI-y')
    ax.quiver(0, 0, ez_x, ez_z, angles='xy', scale_units='xy', scale=1,
              color='g', label='DSI-z')

    setup_ax(ax, 'FAC-x (Radial)', 'FAC-z (Parallel)', '(x, z) plane')

    # ------------- (y, z) plane -------------
    ax = axs[2]
    ax.quiver(0, 0, 1, 0, angles='xy', scale_units='xy', scale=1,
              color='gray',   label='FAC-y')
    ax.quiver(0, 0, 0, 1, angles='xy', scale_units='xy', scale=1,
              color='purple', label='FAC-z')

    ax.quiver(0, 0, ex_y, ex_z, angles='xy', scale_units='xy', scale=1,
              color='r', label='DSI-x')
    ax.quiver(0, 0, ey_y, ey_z, angles='xy', scale_units='xy', scale=1,
              color='b', label='DSI-y')
    ax.quiver(0, 0, ez_y, ez_z, angles='xy', scale_units='xy', scale=1,
              color='g', label='DSI-z')

    setup_ax(ax, 'FAC-y (Longitudinal)', 'FAC-z (Parallel)', '(y, z) plane')

    fig.suptitle(str(da_e_X_DSI_inFAC.time.values[it]), fontsize=14)
    plt.tight_layout()

    # ファイル名：time index をゼロ埋め
    fname = os.path.join(save_dir, f"coord_{frame_idx:06d}.png")
    fig.savefig(fname, dpi=150)
    plt.close(fig)


In [ ]:
n_time = da_e_X_DSI_inFAC.sizes['time']

frame_idx = 0
for it in range(0, n_time, 64*60):  # 64Hz × 60s ごと
    plot_fac_dsi_frame(it, frame_idx, path_base_save_plot)
    frame_idx += 1